In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
with open('/content/drive/MyDrive/Colab Notebooks/GPT2 Task on F1 Data/data/f1_corpus.txt', 'r') as f:
  corpus = f.read()

In [ ]:
print(f"Loaded {len(corpus):,} characters")
print(corpus[:200])

Loaded 9,421,321 characters
Nicolas Hülkenberg (, born 19 August 1987) is a German racing driver who drives for the Haas F1 Team in Formula One. He was the 2009 GP2 Series champion, and is a previous champion of both the Formula


In [ ]:
!pip install transformers datasets wandb -q

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from torch.utils.data import Dataset, DataLoader
import wandb

In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: neil-j-gosalia (neil-j-gosalia-symbiosis-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device}")

cuda


In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenized = tokenizer.encode(corpus)
print(f"Total Tokens: {tokenized}")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
print(f"Total Tokens: {len(tokenized)}")

Total Tokens: 2112318


In [ ]:
type(tokenized)

list

In [ ]:
class GPT2Dataset(Dataset):
  """
  This converts the entire tokenized corpus into a sequence of
  chunks, which cab be fed into the model.
  """
  def __init__(self, tokenized, block_size=1024):
    self.tokenized = tokenized
    self.block_size = block_size
    self.block = [tokenized[1024*i:1024*(i+1)] for i in range(len(tokenized)//block_size)]
  def __len__(self, ):
    return (len(self.block))

  def __getitem__(self, idx):
    return torch.tensor(self.block[idx], dtype=torch.long)

In [ ]:
dataset = GPT2Dataset(tokenized)
split = int(0.9 * len(dataset))
from torch.utils.data import Subset
train_dataset = Subset(dataset, range(split))
val_dataset = Subset(dataset, range(split, len(dataset)))

train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Train batches: {len(train_dataloader)}")
print(f"Val batchesL {len(val_dataloader)}")

Train batches: 464
Val batchesL 52


In [ ]:
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Parameters: 124,439,808


In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
optimizer = AdamW(params=model.parameters(), lr=5e-5, weight_decay=0.01)
num_epochs = 3
total_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=total_steps//10, num_training_steps=total_steps)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {total_steps // 10}")

Total training steps: 1392
Warmup steps: 139


In [ ]:
wandb.init(
    project="gpt2-f1",
    config={
        "model": "gpt2",
        "dataset": "f1_wikipedia",
        "epochs": num_epochs,
        "batch_size": 4,
        "learning_rate": 5e-5,
        "weight_decay": 0.01,
        "block_size": 1024,
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
    }
)

In [ ]:
from torch.nn.utils import clip_grad_norm_

In [ ]:
import os
def save_checkpoint(model, tokenizer, epoch, loss):
  path = f'/content/drive/MyDrive/Colab Notebooks/GPT2 Task on F1 Data/model/checkpoint_epoch_{epoch}'
  os.makedirs(path, exist_ok=True)
  model.save_pretrained(path)
  tokenizer.save_pretrained(path)
  print(f"Checkpoint saved: {path} | Loss: {loss:.4f}")

In [ ]:
from tqdm import tqdm
model.train()
global_step = 0
for epoch in range(num_epochs):

  total_train_loss = 0
  for i, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch+1}")): #batch is a tensor, and 'i' iterates through every tensor
    input_ids = batch.to(device)
    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, labels=input_ids)
    loss = outputs.loss
    loss.backward()
    clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    global_step += 1
    scheduler.step()
    wandb.log({"train_loss":loss.item(), "step":global_step}) #will be recorded for every batch
    total_train_loss += loss.item()
    if i%25==0:
      print(f"Batch: {i} | Loss: {total_train_loss/(i+1)}")
  avg_train_loss = total_train_loss / len(train_dataloader)
  print(f"Training Loss: {avg_train_loss}")

  model.eval()
  with torch.inference_mode():
    total_val_loss = 0
    for batch in tqdm(val_dataloader):
      input_ids = batch.to(device)
      outputs = model(input_ids=input_ids, labels=input_ids)
      total_val_loss += outputs.loss.item()
  model.train()
  avg_val_loss = total_val_loss / len(val_dataloader)
  print(f"Validation Loss: {avg_val_loss}")
  save_checkpoint(model=model,tokenizer=tokenizer, epoch=epoch+1,loss=avg_val_loss)

  wandb.log({"epoch":epoch+1,"avg_train_loss":avg_train_loss,"avg_val_loss":avg_val_loss,"perplexity":torch.exp(torch.tensor(avg_val_loss)).item()})

Epoch 1:   0%|          | 1/464 [00:02<18:22,  2.38s/it]

Batch: 0 | Loss: 3.1645305156707764


Epoch 1:   6%|▌         | 26/464 [00:32<08:56,  1.23s/it]

Batch: 25 | Loss: 3.3100260954636793


Epoch 1:  11%|█         | 51/464 [01:04<08:59,  1.31s/it]

...

Training Loss: 2.5209831639096656


100%|██████████| 52/52 [00:21<00:00,  2.37it/s]

Validation Loss: 2.7143055338125963


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint saved: /content/drive/MyDrive/Colab Notebooks/GPT2 Task on F1 Data/model/checkpoint_epoch_3 | Loss: 2.7143


In [3]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GPT2LMHeadModel.from_pretrained('/content/drive/MyDrive/Colab Notebooks/GPT2 Task on F1 Data/model/checkpoint_epoch_3')
tokenizer = GPT2Tokenizer.from_pretrained('/content/drive/MyDrive/Colab Notebooks/GPT2 Task on F1 Data/model/checkpoint_epoch_3')

model = model.to(device)
model.eval()
print('Model loaded successfully')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded successfully


In [4]:
input_text = "Lewis hamilton enters the final corner"
input_ids = tokenizer.encode(text=input_text, return_tensors='pt').to(device)
with torch.inference_mode():
  output = model.generate(input_ids=input_ids, max_length=100, temperature=0.8, top_p=0.95, do_sample=True, pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Lewis hamilton enters the final corner, the front wing being taken up by the top of the car. The chicane is filled with cars with the front wing blown off by an electrical fault, which would have prevented the car from leaving the track. The car was not damaged during the lap, though, as the car was being driven on the back straight.

Alesi, Häkkinen, and Coulthard are all given red cards. Coulthard, who took the


In [8]:
input_text = "The 2021 formula one world championship"
input_ids = tokenizer.encode(text=input_text, return_tensors='pt').to(device)
with torch.inference_mode():
  output = model.generate(input_ids=input_ids, max_length=100, temperature=0.8, top_p=0.95, do_sample=True, pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(output[0], skip_special_tokens=False))

The 2021 formula one world championship was held in Australia, where it was won by the Red Bull Racing team. It was won by Michael Schumacher, driving a Ferrari F1 car, driving the new Ferrari 312T and Ferrari 312T Hybrid, respectively. Schumacher was a world champion with a win of 21 seconds in his Ferrari 312T, the first victory for a Ferrari car since Jean Alesi in 1984. He was replaced by former teammate Sergio Pérez, who finished second in
